In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

from src.utils.pipeline import load_all_snapshots
from src.features.plate_discipline import add_discipline_flags
from src.features.strike_zone import in_strike_zone
from src.utils.temporal import split_by_date
from src.utils.leakage import banned_for_pitch_outcome, check_features

df = load_all_snapshots()
f = add_discipline_flags(df)
swings = f[f["is_swing"]].copy()
swings["target"] = swings["is_whiff"].astype(int)
swings["in_zone_flag"] = in_strike_zone(swings)

px = pd.to_numeric(swings["plate_x"], errors="coerce").astype("float64")
swings["plate_x_bat"] = np.where(swings["stand"] == "L", -px, px)
pz = pd.to_numeric(swings["plate_z"], errors="coerce").astype("float64")
top = pd.to_numeric(swings["sz_top"], errors="coerce").astype("float64")
bot = pd.to_numeric(swings["sz_bot"], errors="coerce").astype("float64")
swings["plate_z_rel"] = (pz - bot) / (top - bot)

split = split_by_date(swings, train_end="2024-07-14", validation_end="2024-08-31")

NUMERIC = ["plate_x_bat", "plate_z_rel", "release_speed", "pfx_x", "pfx_z",
           "release_spin_rate", "release_extension", "balls", "strikes"]

counts = split.train["pitch_type"].value_counts()
common = set(counts[counts >= 500].index)
for part in [split.train, split.validation, split.test]:
    pt = part["pitch_type"].astype(str)
    part["pitch_type_b"] = pt.where(pt.isin(common), "OTHER")

def build_X(part):
    num = part[NUMERIC].apply(pd.to_numeric, errors="coerce").astype("float64")
    cat = pd.get_dummies(part[["pitch_type_b", "stand", "p_throws"]].astype(str),
                         drop_first=True)
    X = pd.concat([num, cat], axis=1)
    pzv = pd.to_numeric(part["plate_z_rel"], errors="coerce").astype("float64")
    for col in [c for c in X.columns if c.startswith("pitch_type_b_")]:
        X[f"{col}_x_pz"] = X[col].astype(float) * pzv.to_numpy()
    return X

X_train = build_X(split.train)
X_val = build_X(split.validation).reindex(columns=X_train.columns, fill_value=0)
y_train = split.train["target"].to_numpy()
y_val = split.validation["target"].to_numpy()

check_features(X_train.columns, banned_for_pitch_outcome(), context="whiff logistic D")

model_d = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=42)),
]).fit(X_train, y_train)

# baseline
def fit_lookup(train, keys, prior_strength=50.0):
    g = train["target"].mean()
    t = train.groupby(keys)["target"].agg(["sum", "size"])
    return ((t["sum"] + g * prior_strength) / (t["size"] + prior_strength)), g

def predict_lookup(part, table, fallback, keys):
    return table.reindex(pd.MultiIndex.from_frame(part[keys])).fillna(fallback).to_numpy()

LOOKUP2_KEYS = ["pitch_type", "balls", "strikes", "in_zone_flag"]
lookup2, global2 = fit_lookup(split.train, LOOKUP2_KEYS)

print("ready")

ready


In [3]:
from sklearn.calibration import calibration_curve

p_val = model_d.predict_proba(X_val)[:, 1]
p_base = predict_lookup(split.validation, lookup2, global2, LOOKUP2_KEYS)

def reliability(y, p, bins=10):
    frac_pos, mean_pred = calibration_curve(y, p, n_bins=bins, strategy="quantile")
    edges = np.quantile(p, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    idx = np.digitize(p, edges[1:-1])
    counts = [int((idx == b).sum()) for b in range(bins)]
    return pd.DataFrame({
        "predicted": mean_pred,
        "actual": frac_pos,
        "gap": frac_pos - mean_pred,
        "n": counts[:len(mean_pred)],
    })

print("=== logistic D ===")
print(reliability(y_val, p_val).round(4).to_string(index=False))
print()
print("=== lookup baseline ===")
print(reliability(y_val, p_base).round(4).to_string(index=False))

=== logistic D ===
 predicted  actual     gap    n
    0.0617  0.1163  0.0546 8350
    0.0970  0.1003  0.0033 8349
    0.1213  0.1091 -0.0122 8349
    0.1459  0.1215 -0.0244 8349
    0.1743  0.1685 -0.0058 8349
    0.2075  0.1940 -0.0135 8349
    0.2484  0.2162 -0.0322 8349
    0.3014  0.2738 -0.0276 8349
    0.3839  0.3654 -0.0185 8349
    0.5829  0.6468  0.0639 8350

=== lookup baseline ===
 predicted  actual     gap    n
    0.0885  0.0821 -0.0064 6998
    0.1227  0.1252  0.0025 9382
    0.1463  0.1387 -0.0076 8542
    0.1610  0.1546 -0.0064 8275
    0.1734  0.1714 -0.0020 6915
    0.1809  0.1717 -0.0092 9769
    0.2126  0.2094 -0.0032 7279
    0.2810  0.2933  0.0123 9549
    0.3846  0.4094  0.0248 8431
    0.5488  0.5703  0.0215 8352


In [4]:
def ece(y, p, bins=10):
    """Expected Calibration Error: sample-weighted mean |actual - predicted|."""
    edges = np.quantile(p, np.linspace(0, 1, bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf
    idx = np.digitize(p, edges[1:-1])
    total = 0.0
    for b in range(bins):
        m = idx == b
        if m.sum() == 0:
            continue
        total += m.sum() * abs(y[m].mean() - p[m].mean())
    return total / len(y)

print(f"ECE logistic D: {ece(y_val, p_val):.5f}")
print(f"ECE baseline:   {ece(y_val, p_base):.5f}")

ECE logistic D: 0.02560
ECE baseline:   0.00903


In [ ]:
from sklearn.frozen import FrozenEstimator
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score

n_cal = len(X_val) // 2
X_cal, y_cal = X_val.iloc[:n_cal], y_val[:n_cal]
X_eval, y_eval = X_val.iloc[n_cal:], y_val[n_cal:]

# FrozenEstimator replaces cv="prefit", removed in recent sklearn.
# It wraps an already-fitted model so the calibrator does not refit it.
cal_iso = CalibratedClassifierCV(FrozenEstimator(model_d), method="isotonic")
cal_iso.fit(X_cal, y_cal)

cal_sig = CalibratedClassifierCV(FrozenEstimator(model_d), method="sigmoid")
cal_sig.fit(X_cal, y_cal)

p_raw = model_d.predict_proba(X_eval)[:, 1]
p_iso = cal_iso.predict_proba(X_eval)[:, 1]
p_sig = cal_sig.predict_proba(X_eval)[:, 1]
p_base_eval = predict_lookup(split.validation.iloc[n_cal:], lookup2, global2, LOOKUP2_KEYS)

for name, p in [("logistic raw", p_raw),
                ("logistic isotonic", p_iso),
                ("logistic sigmoid", p_sig),
                ("lookup baseline", p_base_eval)]:
    print(f"{name:20s} log_loss {log_loss(y_eval, p):.5f}  "
          f"brier {brier_score_loss(y_eval, p):.5f}  "
          f"auc {roc_auc_score(y_eval, p):.5f}  "
          f"ECE {ece(y_eval, p):.5f}")

logistic raw         log_loss 0.46854  brier 0.14819  auc 0.73145  ECE 0.02776
logistic isotonic    log_loss 0.46449  brier 0.14694  auc 0.73090  ECE 0.00883
logistic sigmoid     log_loss 0.46880  brier 0.14835  auc 0.73145  ECE 0.03004
lookup baseline      log_loss 0.48102  brier 0.15451  auc 0.71205  ECE 0.01044
